In [1]:
from pyspark.sql import functions as F
from datetime import datetime

spark.sql("CREATE SCHEMA IF NOT EXISTS audit")

batch_id = datetime.now().strftime("%Y%m%d%H%M%S")
run_start_time = datetime.now()

print(f"Audit monitoring started. Batch ID: {batch_id}")

StatementMeta(, 40e8f09d-ca08-40a9-8872-58f8f441f477, 3, Finished, Available, Finished, False)

Audit monitoring started. Batch ID: 20260717010032


In [2]:
def write_delta_append(df, table_name):
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"Appended {df.count()} rows into {table_name}")


def safe_table_count(table_name):
    try:
        return spark.table(table_name).count()
    except Exception:
        return None

StatementMeta(, 40e8f09d-ca08-40a9-8872-58f8f441f477, 4, Finished, Available, Finished, False)

In [3]:
tables_to_monitor = [
    # Bronze
    ("bronze", "orders"),
    ("bronze", "customers"),
    ("bronze", "products"),
    ("bronze", "inventory"),
    ("bronze", "returns"),
    ("bronze", "clickstream"),
    ("bronze", "stores"),

    # Silver
    ("silver", "orders"),
    ("silver", "customers"),
    ("silver", "products"),
    ("silver", "inventory"),
    ("silver", "returns"),
    ("silver", "clickstream"),
    ("silver", "stores"),

    # Quarantine
    ("quarantine", "orders"),
    ("quarantine", "customers"),
    ("quarantine", "products"),
    ("quarantine", "inventory"),
    ("quarantine", "returns"),
    ("quarantine", "clickstream"),
    ("quarantine", "stores"),

    # Gold
    ("gold", "dim_customer"),
    ("gold", "dim_product"),
    ("gold", "dim_store"),
    ("gold", "dim_channel"),
    ("gold", "dim_date"),
    ("gold", "fact_sales"),
    ("gold", "fact_returns"),
    ("gold", "fact_inventory_snapshot")
]

row_count_records = []

for schema_name, table_name in tables_to_monitor:
    full_table_name = f"{schema_name}.{table_name}"
    row_count = safe_table_count(full_table_name)

    row_count_records.append({
        "batch_id": batch_id,
        "audit_timestamp": datetime.now(),
        "schema_name": schema_name,
        "table_name": table_name,
        "full_table_name": full_table_name,
        "row_count": row_count
    })

row_count_df = spark.createDataFrame(row_count_records)

write_delta_append(row_count_df, "audit.table_row_counts")

display(row_count_df.orderBy("schema_name", "table_name"))

StatementMeta(, 40e8f09d-ca08-40a9-8872-58f8f441f477, 5, Finished, Available, Finished, False)

Appended 29 rows into audit.table_row_counts


SynapseWidget(Synapse.DataFrame, c6475156-6942-4aad-99d8-8a51806d5d94)

In [4]:
quarantine_tables = [
    "quarantine.orders",
    "quarantine.customers",
    "quarantine.products",
    "quarantine.inventory",
    "quarantine.returns",
    "quarantine.clickstream",
    "quarantine.stores"
]

dq_summary_dfs = []

for table_name in quarantine_tables:
    try:
        df = spark.table(table_name)

        if "_dq_error" in df.columns:
            summary_df = (
                df
                .groupBy("_dq_error")
                .agg(F.count("*").alias("rejected_record_count"))
                .withColumn("batch_id", F.lit(batch_id))
                .withColumn("audit_timestamp", F.current_timestamp())
                .withColumn("quarantine_table", F.lit(table_name))
                .select(
                    "batch_id",
                    "audit_timestamp",
                    "quarantine_table",
                    "_dq_error",
                    "rejected_record_count"
                )
            )

            dq_summary_dfs.append(summary_df)

    except Exception as e:
        print(f"Could not process {table_name}: {e}")

if dq_summary_dfs:
    dq_summary = dq_summary_dfs[0]

    for df in dq_summary_dfs[1:]:
        dq_summary = dq_summary.unionByName(df)

    write_delta_append(dq_summary, "audit.data_quality_summary")

    display(
        dq_summary
        .orderBy(F.desc("rejected_record_count"))
    )
else:
    print("No quarantine data quality errors found.")

StatementMeta(, 40e8f09d-ca08-40a9-8872-58f8f441f477, 6, Finished, Available, Finished, False)

Appended 12 rows into audit.data_quality_summary


SynapseWidget(Synapse.DataFrame, 8fdbdabf-f857-4399-9b52-649f0a3ec562)

In [5]:
validation_records = []

def add_validation_check(check_name, table_name, failed_count, severity):
    validation_records.append({
        "batch_id": batch_id,
        "audit_timestamp": datetime.now(),
        "check_name": check_name,
        "table_name": table_name,
        "failed_count": int(failed_count),
        "severity": severity,
        "status": "PASS" if failed_count == 0 else "FAIL"
    })


# fact_sales checks
fact_sales = spark.table("gold.fact_sales")

add_validation_check(
    "fact_sales_missing_customer_sk",
    "gold.fact_sales",
    fact_sales.filter(F.col("customer_sk").isNull()).count(),
    "HIGH"
)

add_validation_check(
    "fact_sales_missing_product_sk",
    "gold.fact_sales",
    fact_sales.filter(F.col("product_sk").isNull()).count(),
    "HIGH"
)

add_validation_check(
    "fact_sales_missing_store_sk",
    "gold.fact_sales",
    fact_sales.filter(F.col("store_sk").isNull()).count(),
    "MEDIUM"
)

add_validation_check(
    "fact_sales_negative_net_sales",
    "gold.fact_sales",
    fact_sales.filter(F.col("net_sales_amount") < 0).count(),
    "HIGH"
)

add_validation_check(
    "fact_sales_missing_order_id",
    "gold.fact_sales",
    fact_sales.filter(F.col("order_id").isNull()).count(),
    "HIGH"
)


# fact_returns checks
fact_returns = spark.table("gold.fact_returns")

add_validation_check(
    "fact_returns_missing_customer_sk",
    "gold.fact_returns",
    fact_returns.filter(F.col("customer_sk").isNull()).count(),
    "MEDIUM"
)

add_validation_check(
    "fact_returns_missing_product_sk",
    "gold.fact_returns",
    fact_returns.filter(F.col("product_sk").isNull()).count(),
    "MEDIUM"
)

add_validation_check(
    "fact_returns_negative_refund",
    "gold.fact_returns",
    fact_returns.filter(F.col("refund_amount") < 0).count(),
    "HIGH"
)


# inventory checks
fact_inventory = spark.table("gold.fact_inventory_snapshot")

add_validation_check(
    "inventory_missing_product_sk",
    "gold.fact_inventory_snapshot",
    fact_inventory.filter(F.col("product_sk").isNull()).count(),
    "HIGH"
)

add_validation_check(
    "inventory_missing_store_sk",
    "gold.fact_inventory_snapshot",
    fact_inventory.filter(F.col("store_sk").isNull()).count(),
    "HIGH"
)

add_validation_check(
    "inventory_negative_available_qty",
    "gold.fact_inventory_snapshot",
    fact_inventory.filter(F.col("available_qty") < 0).count(),
    "MEDIUM"
)


validation_df = spark.createDataFrame(validation_records)

write_delta_append(validation_df, "audit.gold_validation_summary")

display(validation_df.orderBy("severity", "check_name"))

StatementMeta(, 40e8f09d-ca08-40a9-8872-58f8f441f477, 7, Finished, Available, Finished, False)

Appended 11 rows into audit.gold_validation_summary


SynapseWidget(Synapse.DataFrame, 6df7c097-b27f-4308-a7c9-a3504e43941b)

In [6]:
run_end_time = datetime.now()
duration_seconds = int((run_end_time - run_start_time).total_seconds())

total_bronze_rows = sum([
    r["row_count"] or 0
    for r in row_count_records
    if r["schema_name"] == "bronze"
])

total_silver_rows = sum([
    r["row_count"] or 0
    for r in row_count_records
    if r["schema_name"] == "silver"
])

total_quarantine_rows = sum([
    r["row_count"] or 0
    for r in row_count_records
    if r["schema_name"] == "quarantine"
])

total_gold_rows = sum([
    r["row_count"] or 0
    for r in row_count_records
    if r["schema_name"] == "gold"
])

failed_high_checks = (
    validation_df
    .filter((F.col("status") == "FAIL") & (F.col("severity") == "HIGH"))
    .count()
)

pipeline_status = "SUCCESS" if failed_high_checks == 0 else "SUCCESS_WITH_WARNINGS"

pipeline_log = [{
    "batch_id": batch_id,
    "pipeline_name": "manual_retailops_end_to_end_run",
    "run_start_time": run_start_time,
    "run_end_time": run_end_time,
    "duration_seconds": duration_seconds,
    "status": pipeline_status,
    "total_bronze_rows": total_bronze_rows,
    "total_silver_rows": total_silver_rows,
    "total_quarantine_rows": total_quarantine_rows,
    "total_gold_rows": total_gold_rows,
    "high_severity_failed_checks": failed_high_checks
}]

pipeline_log_df = spark.createDataFrame(pipeline_log)

write_delta_append(pipeline_log_df, "audit.pipeline_run_log")

display(pipeline_log_df)

StatementMeta(, 40e8f09d-ca08-40a9-8872-58f8f441f477, 8, Finished, Available, Finished, False)

Appended 1 rows into audit.pipeline_run_log


SynapseWidget(Synapse.DataFrame, 202dc18e-ab0d-446a-acbb-89b2bd158e67)

In [7]:
display(spark.table("audit.pipeline_run_log").orderBy(F.desc("run_start_time")))

StatementMeta(, 40e8f09d-ca08-40a9-8872-58f8f441f477, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 48564ab3-8054-44fe-b181-ec228cf9fb71)

In [8]:
display(
    spark.table("audit.data_quality_summary")
    .groupBy("quarantine_table")
    .agg(F.sum("rejected_record_count").alias("total_rejected_records"))
    .orderBy(F.desc("total_rejected_records"))
)

StatementMeta(, 40e8f09d-ca08-40a9-8872-58f8f441f477, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1dae81b9-ca0d-435c-aa5b-7e5e6b3229cb)